In [46]:
import os.path
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import keras
import librosa
from keras import Sequential
from keras.src.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from keras.src.utils import audio_dataset_from_directory, to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [3]:
SAMPLE_RATE = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512

In [4]:
metadata = pd.read_csv('data/ESC-50-master/meta/esc50.csv')

In [5]:
def load_audio(path:str,sr=SAMPLE_RATE, duration=DURATION) :
    audio,src = librosa.load(path,sr=SAMPLE_RATE,duration=DURATION)
    return audio


In [6]:
def extract_stft_features(audio,n_fft,hop_length):
    stft = librosa.stft(audio,n_fft=n_fft,hop_length=hop_length)
    magnitude = np.abs(stft)
    stft_db = librosa.amplitude_to_db(magnitude,ref=np.max)
    return stft_db

In [7]:
def load_dataset() :
    features = []
    labels = []

    for index,row in tqdm(metadata.iterrows(),total=len(metadata)):
        file_name = row['filename']
        category = row['category']

        audio = os.path.join('data/ESC-50-master/audio/', file_name)
        audio = load_audio(audio, SAMPLE_RATE, DURATION)
        if audio is not None :
            stft = extract_stft_features(audio,n_fft=N_FFT,hop_length=HOP_LENGTH)
            features.append(stft)
            labels.append(category)
    features = np.array(features)
    labels = np.array(labels)
    return features,labels

In [22]:
features,labels =load_dataset()

100%|██████████| 2000/2000 [00:13<00:00, 143.08it/s]


In [50]:
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_c = to_categorical(labels_encoded)
labels = labels_c

In [51]:
x_train,x_test,y_train,y_test = train_test_split(features,labels,test_size=0.2,random_state=42,stratify=labels)

In [ ]:
plt.plot(x_test[0])
plt.show()


In [37]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),activation='relu',input_shape=(1025,216,1)))
model.add(Conv2D(64, (3, 3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(50, activation='softmax'))

In [52]:
model.compile(optimizer='adam',loss=keras.losses.categorical_crossentropy,metrics=['accuracy'])
print(x_train.dtype)
print(y_train.dtype)

print(x_train.shape)
print(y_train.shape)

float32
float64
(1600, 1025, 216)
(1600, 50)


In [ ]:
model.fit(x_train,y_train,batch_size=64,epochs=10,validation_data=(x_test,y_test))

Epoch 1/10
